### Install Dependencies and Import Dependencies

- UXsim (simulator)
- gymnasium (environment class)
- torch (neural networks and ML stuff)
- pandas (dataframes)
- numpy (fast numerical computation)

In [ ]:
!pip install uxsim gymnasium torch pandas numpy

### Ensure that Classes can reference their own types in type hint signatures

In [ ]:
from __future__ import annotations

### Import Dependencies

In [ ]:
import matplotlib.pyplot
import pandas
import numpy
import gymnasium
import uxsim
import random
from collections import namedtuple, deque
import math
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import os

### Episode-level and Epoch-level tracking for features and metrics

In [ ]:
class Episode:
  def __init__(self) -> None:
    self.step = []
    self.total_reward = []

  def record(self, total_reward: float):
    self.step.append(len(self.step) + 1)
    self.total_reward.append(total_reward)

  def to_df(self) -> pandas.DataFrame:
    return pandas.DataFrame({
      "Step": self.step,
      "TotalReward": self.total_reward
    })

class History:
  def __init__(self) -> None:
    self.episode = []
    self.total_reward = []

  def record(self, total_reward: float):
    self.episode.append(len(self.episode) + 1)
    self.total_reward.append(total_reward)

  def to_df(self) -> pandas.DataFrame:
    return pandas.DataFrame({
      "Episode": self.episode,
      "TotalReward": self.total_reward
    })

### Simulator class using UXsim and gymnasium.Env

In [ ]:
class Simulator(gymnasium.Env):
  def __init__(self, fEW: float, fWE: float, fNS: float, fSN: float):
    """
    traffic scenario: 4-legged intersection with 2 phase signal.
    action 1: greenlight for direction 1
    action 2: greenlight for direction 2
    state: number of waiting vehicles for each incoming link
    reward: negative of difference of total waiting vehicles
    """

    # Platoons, Delta-Time and Duration
    self.deltan = 1
    self.delta_time = 10
    self.duration = 3600

    # Topology
    self.lanes_per_road = 2
    self.jam_density = 0.2
    self.road_length = 1000
    self.max_allowed_speed = 20

    # Action Space
    self.n_action = 2
    self.action_space = gymnasium.spaces.Discrete(self.n_action)

    # State Space
    self.n_state = 4
    low = numpy.array([0 for _ in range(self.n_state)])
    high = numpy.array([1 for _ in range(self.n_state)])
    self.observation_space = gymnasium.spaces.Box(low=low, high=high)

    self.fEW: float = fEW
    self.fWE: float = fWE
    self.fNS: float = fNS
    self.fSN: float = fSN

    self.reset()

  def total_vehicles(self) -> int:
    return len(self.W.VEHICLES)

  def reset(self, *, seed: int|None = None, options: dict|None = None):
    """
    reset the env
    """
    seed = None #demand is always random
    if options is None:
      options = {}
    vehicle_logging_timestep_interval=-1
    if 'gif' in options:
      vehicle_logging_timestep_interval=1
    self.W = uxsim.World(
      name="",
      deltan=self.deltan,
      tmax=self.duration,
      print_mode=True,
      save_mode=False,
      show_mode=True,
      reduce_memory_delete_vehicle_route_pref=True,
      vehicle_logging_timestep_interval=vehicle_logging_timestep_interval,
      random_seed=seed,
      duo_update_time=99999
    )
    random.seed(seed)

    #network definition
    self.II = self.W.addNode("Intersection", 0, 0, signal=[60,60])
    self.EE = self.W.addNode("E", 1, 0)
    self.WW = self.W.addNode("W", -1, 0)
    self.SS = self.W.addNode("S", 0, 1)
    self.NN = self.W.addNode("N", 0, -1)
    self.W.addLink("EI", self.EE, self.II, number_of_lanes=self.lanes_per_road, length=self.road_length, free_flow_speed=self.max_allowed_speed, jam_density=self.jam_density, signal_group=[0])
    self.W.addLink("WI", self.WW, self.II, number_of_lanes=self.lanes_per_road, length=self.road_length, free_flow_speed=self.max_allowed_speed, jam_density=self.jam_density, signal_group=[0])
    self.W.addLink("SI", self.SS, self.II, number_of_lanes=self.lanes_per_road, length=self.road_length, free_flow_speed=self.max_allowed_speed, jam_density=self.jam_density, signal_group=[1])
    self.W.addLink("NI", self.NN, self.II, number_of_lanes=self.lanes_per_road, length=self.road_length, free_flow_speed=self.max_allowed_speed, jam_density=self.jam_density, signal_group=[1])
    self.W.addLink("IE", self.II, self.EE, number_of_lanes=self.lanes_per_road, length=self.road_length, free_flow_speed=self.max_allowed_speed, jam_density=self.jam_density)
    self.W.addLink("IW", self.II, self.WW, number_of_lanes=self.lanes_per_road, length=self.road_length, free_flow_speed=self.max_allowed_speed, jam_density=self.jam_density)
    self.W.addLink("IS", self.II, self.SS, number_of_lanes=self.lanes_per_road, length=self.road_length, free_flow_speed=self.max_allowed_speed, jam_density=self.jam_density)
    self.W.addLink("IN", self.II, self.NN, number_of_lanes=self.lanes_per_road, length=self.road_length, free_flow_speed=self.max_allowed_speed, jam_density=self.jam_density)

    # # random demand definition
    self.W.adddemand(self.EE, self.WW, 0, self.duration, self.fEW)
    self.W.adddemand(self.WW, self.EE, 0, self.duration, self.fWE)
    self.W.adddemand(self.NN, self.SS, 0, self.duration, self.fNS)
    self.W.adddemand(self.SS, self.NN, 0, self.duration, self.fSN)

    #initial observation
    observation = numpy.array([0 for _ in range(self.n_state)])

    #log
    self.enable_log = False
    self.log_state = []
    self.log_action = []
    self.log_reward = []

    self.compute_metrics()
    self.old_num_vehicles_queue = self.num_vehicles_queue
    return observation, {}

  def mean(self, iterator):
    summa = 0
    count = 0
    for el in iterator:
      summa += el
      count += 1
    if count > 0:
      summa /= count
    return summa

  def compute_metrics(self):
    """
    compute the metrics for the current state and reward
    """
    self.vehicles_per_links = {}
    self.num_vehicles_queue = 0
    for l in self.II.inlinks.values():
      self.vehicles_per_links[l] = l.num_vehicles_queue
      self.num_vehicles_queue += l.num_vehicles_queue

    # self.average_speed_per_links = {}
    # for l in self.W.LINKS:
    #   self.average_speed_per_links[l] = self.mean(map(lambda v: v.v, l.vehicles))
    # self.average_speed = self.mean(self.average_speed_per_links.values())

  def compute_state(self) -> numpy.ndarray:
    """
    compute the current state
    """
    state = numpy.array(list(self.vehicles_per_links.values()))
    if self.num_vehicles_queue > 0:
      state = state / self.num_vehicles_queue
    return state

  def compute_reward(self) -> float:
    """
    compute the reward for the last action
    """
    # return self.average_speed - 10
    return self.old_num_vehicles_queue - self.num_vehicles_queue

  def step(self, action):
    """
    proceed env by 1 step = K seconds
    """

    #change signal by action
    assert action >= 0 and action < self.n_action
    self.II.signal_phase = action
    self.II.signal_t = 0

    #traffic dynamics. execute simulation for 30 seconds
    if self.W.check_simulation_ongoing():
      self.W.exec_simulation(duration_t2=self.delta_time)

    self.compute_metrics()

    #observe state
    observation = self.compute_state()

    #compute reward
    reward = self.compute_reward()

    self.old_num_vehicles_queue = self.num_vehicles_queue

    #check termination
    done = False
    if not self.W.check_simulation_ongoing():
      done = True
      observation = None

    #log
    if self.enable_log:
      self.log_state.append(observation)
      self.log_action.append(action)
      self.log_reward.append(reward)

    return observation, reward, done, done, {}

# Classical Traffic Light Schedule Agent

In [ ]:
class ClassicTrafficLightAgent:
  def __init__(self, number_of_phases: int, phase_duration: int):
    self.number_of_phases = number_of_phases
    # Duration in number of ticks
    self.phase_duration = phase_duration
    self.current_phase: int = 0
    self.phase_timer: int = 0

  def choose(self, _) -> int:
    if self.phase_timer > self.phase_duration:
      self.current_phase = ((self.current_phase + 1) % self.number_of_phases)
      self.phase_timer = 0
    else:
      self.phase_timer += 1
    return self.current_phase

  def reset(self):
    self.current_phase = 0
    self.phase_timer = 0

CTLAs = {
  ('CTLA_%s0s'%v): ClassicTrafficLightAgent(2, v)
  for v in range(1, 7)
}

### Stuff for DQNs

- Replay Memory
- DQN (Neural Network)
- Model (DQN implementation)

In [ ]:
# the number of transitions sampled from the replay buffer
BATCH_SIZE = 512
# the size of replay memory
MEM_SIZE = 10000
# the size of networks
NET_SIZE = 32
# the depth of networks
NET_DEPTH = 2
# the discount factor as mentioned in the previous section
GAMMA = 0.99
# the starting value of epsilon
EPS_START = 0.9
# the final value of epsilon
EPS_END = 0.05
# the rate of exponential decay of epsilon, higher means a slower decay
EPS_DECAY = 1000
# the update rate of the target network
TAU = 0.005
# the learning rate of the ``AdamW`` optimizer
LR = 1e-4

Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward'))

class ReplayMemory:
  def __init__(self, capacity):
    self.memory = deque([], maxlen=capacity)

  def push(self, *args):
    """Save a transition"""
    self.memory.append(Transition(*args))

  def sample(self, batch_size):
    return random.sample(self.memory, batch_size)

  def __len__(self):
    return len(self.memory)

class DQN(nn.Module):
  def __init__(self, n_observations, n_actions):
    super().__init__()
    n_neurals = NET_SIZE
    n_layers = NET_DEPTH
    self.layers = nn.ModuleList()
    self.layers.append(nn.Linear(n_observations, n_neurals))
    for _ in range(n_layers):
      self.layers.append(nn.Linear(n_neurals, n_neurals))
    self.layer_last = nn.Linear(n_neurals, n_actions)

  # Called with either one element to determine next action, or a batch during optimization. Returns tensor([[left0exp,right0exp]...]).
  def forward(self, x):
    for layer in self.layers:
      x = F.relu(layer(x))
    return self.layer_last(x)

  def save(self, path: str):
    torch.save(self.state_dict(), path)

  def load(self, path: str):
    self.load_state_dict(torch.load(path))

  def import_average_of(self, dqns: list[tuple[float, DQN]]):
    state_dicts = [dqn[1].state_dict() for dqn in dqns]
    weights = [dqn[0] for dqn in dqns]
    avg_state_dict = {
      key: state_dicts[0][key] * weights[0] for key in state_dicts[0]
    }
    for idx in range(1, len(state_dicts)):
      for key in avg_state_dict:
        avg_state_dict[key] = avg_state_dict[key] + state_dicts[idx][key] * weights[idx]
    self.load_state_dict(avg_state_dict)

class Model:
  def __init__(self, n_observations: int, n_actions: int, policy_net: DQN, target_net: DQN) -> None:
    self.n_observations = n_observations
    self.n_actions = n_actions
    self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    self.policy_net = policy_net.to(self.device)
    self.target_net = target_net.to(self.device)
    self.memory = ReplayMemory(MEM_SIZE)
    self.optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
    self.steps_done = 0

  def import_average_of(self, models: list[tuple[float, Model]]):
    self.policy_net.import_average_of([(weight, model.policy_net) for (weight, model) in models])
    self.target_net.import_average_of([(weight, model.target_net) for (weight, model) in models])

  def load_state_of(self, other: Model):
    self.policy_net.load_state_dict(other.policy_net.state_dict())
    self.target_net.load_state_dict(other.target_net.state_dict())

  @staticmethod
  def create(n_observations: int, n_actions: int):
    policy_net = DQN(n_observations, n_actions)
    target_net = DQN(n_observations, n_actions)
    target_net.load_state_dict(policy_net.state_dict())
    return Model(n_observations, n_actions, policy_net, target_net)

  def predict(self, state):
    state = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
    sample = random.random()
    eps_threshold = EPS_END + (EPS_START - EPS_END) * math.exp(-1. * self.steps_done / EPS_DECAY)
    self.steps_done += 1
    if sample > eps_threshold:
      with torch.no_grad():
        # t.max(1) will return the largest column value of each row. second column on max result is index of where max element was found, so we pick action with the larger expected reward.
        return self.policy_net(state).max(1)[1].view(1, 1)
    else:
      return torch.tensor([[random.randint(0, self.n_actions - 1)]], device=self.device, dtype=torch.long)

  def add_to_memory(self, state, action, next_state, reward):
    state = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
    if next_state is not None:
      next_state = torch.tensor(next_state, dtype=torch.float32, device=self.device).unsqueeze(0)
    reward = torch.tensor([reward], device=self.device)
    self.memory.push(state, action, next_state, reward)

  def optimize_model(self):
    if len(self.memory) < BATCH_SIZE:
      return
    transitions = self.memory.sample(BATCH_SIZE)
    # Transpose the batch (see https://stackoverflow.com/a/19343/3343043 for detailed explanation). This converts batch-array of Transitions to Transition of batch-arrays.
    batch = Transition(*zip(*transitions))

    # Compute a mask of non-final states and concatenate the batch elements (a final state would've been the one after which simulation ended)
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None, batch.next_state)), device=self.device, dtype=torch.bool)
    non_final_next_states = torch.cat([s for s in batch.next_state if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    # Compute Q(s_t, a) - the model computes Q(s_t), then we select the columns of actions taken. These are the actions which would've been taken for each batch state according to policy_net
    state_action_values = self.policy_net(state_batch).gather(1, action_batch)

    # Compute V(s_{t+1}) for all next states.
    # Expected values of actions for non_final_next_states are computed based on the "older" target_net; selecting their best reward with max(1)[0].
    # This is merged based on the mask, such that we'll have either the expected state value or 0 in case the state was final.
    next_state_values = torch.zeros(BATCH_SIZE, device=self.device)
    with torch.no_grad():
        next_state_values[non_final_mask] = self.target_net(non_final_next_states).max(1)[0]
    # Compute the expected Q values
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    # Compute Huber loss
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    # Optimize the model
    self.optimizer.zero_grad()
    loss.backward()
    # In-place gradient clipping
    torch.nn.utils.clip_grad_value_(self.policy_net.parameters(), 100)
    self.optimizer.step()

    # Soft update of the target network's weights
    # θ′ ← τ θ + (1 −τ )θ′
    target_net_state_dict = self.target_net.state_dict()
    policy_net_state_dict = self.policy_net.state_dict()
    for key in policy_net_state_dict:
      target_net_state_dict[key] = policy_net_state_dict[key]*TAU + target_net_state_dict[key]*(1-TAU)
    self.target_net.load_state_dict(target_net_state_dict)

  def save(self, path: str):
    self.policy_net.save(os.path.join(path, "policy_net"))
    self.target_net.save(os.path.join(path, "target_net"))

  def load(self, path: str):
    self.policy_net.load(os.path.join(path, "policy_net"))
    self.target_net.load(os.path.join(path, "target_net"))

### Utility functions for learning and simulating

- `learn(env, model)` a model learns for 1 episode with an environment.
- `learn_ex(envs, model)` a model learns for 1 episode with each one of the  environments. The metrics are the average of the ones obtained in the multiple runs.
- `evaluate(env, selector)` An episode with the environment is run and the action is chosen with the selector function (accepting the state).
- `evaluate_ex(env, selector)` An episode with each one of the environments is run and the actions are chosen with the selector function (accepting the state). The metrics are the average of the ones obtained in the multiple runs.

In [ ]:
def learn(env: Simulator, model: Model) -> Episode:
  episode = Episode()
  total_reward = 0
  state, _ = env.reset()
  while True:
    action = model.predict(state)
    next_state, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward
    model.add_to_memory(state, action, next_state, reward)
    model.optimize_model()
    if terminated or truncated:
      break
    state = next_state
  episode.record(total_reward)
  env.reset()
  return episode

def evaluate(env: Simulator, selector) -> Episode:
  episode = Episode()
  total_reward = 0
  state, _ = env.reset()
  while True:
    action = selector(state)
    state, reward, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
      break
    total_reward += reward
  episode.record(total_reward)
  env.reset()
  return episode

In [ ]:
def learn_ex(envs: list[Simulator], model: Model) -> Episode:
  assert len(envs) > 0
  episode = Episode()
  total_reward = 0
  queue_of_envs = [env for env in envs]
  random.shuffle(queue_of_envs)
  queue_of_states = [env.reset()[0] for env in queue_of_envs]
  while len(queue_of_envs) > 0:
    new_queue_of_envs = []
    new_queue_of_states = []
    round_reward = 0
    for (env, state) in zip(queue_of_envs, queue_of_states):
      action = model.predict(state)
      next_state, reward, terminated, truncated, _ = env.step(action)
      round_reward += reward
      if not terminated and not truncated:
        new_queue_of_envs.append(env)
        new_queue_of_states.append(next_state)
        model.add_to_memory(state, action, next_state, reward)
    total_reward += (round_reward / len(queue_of_envs))
    model.optimize_model()
    queue_of_states = new_queue_of_states
    queue_of_envs = new_queue_of_envs
  episode.record(total_reward)
  for env in envs:
    env.reset()
  return episode

def evaluate_ex(envs: list[Simulator], selector) -> Episode:
  assert len(envs) > 0
  episode = Episode()
  total_reward = 0
  for env in envs:
    sub_episode = evaluate(env, selector)
    total_reward += sub_episode.total_reward[-1]
  total_reward /= len(envs)
  episode.record(total_reward)
  return episode

In [ ]:
def trace_demo(env: Simulator, selector) -> Episode:
  episode = Episode()
  total_reward = 0
  state, _ = env.reset(options={'gif': True})
  while True:
    action = selector(state)
    state, reward, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
      break
    total_reward += reward
  episode.record(total_reward)
  return episode

### Setup of Environments for Training and Evaluation

The training environments are set to the following scenarious:

- All Mid (Piazzale Fratelli Zavattari)
- Peak East to West (Piazzale Loreto)
- Peak West to East (Piazza Simone Bolivar)
- Peak North to South (Piazza Firenze)
- Peak South to North (Piazzale Lodi)

The evaluation environments are generated randomly sampling for each direction a decimal flow number between $0.3$ and $0.8$.

In [ ]:
LOW = 0.3
MID = 0.5
PEAK = 0.8

training_confs = [
  [ MID,  MID,  MID,  MID], # Mid for All
  [PEAK,  MID, PEAK,  MID], # Peak East to West
  [ MID, PEAK, PEAK,  MID], # Peak North to South
  [PEAK,  MID,  MID, PEAK], # Peak East to West
  [ MID, PEAK,  MID, PEAK], # Peak North to South
]
evaluation_confs = [
  [random.uniform(LOW, PEAK),
   random.uniform(LOW, PEAK),
   random.uniform(LOW, PEAK),
   random.uniform(LOW, PEAK)]
  for _ in range(5)
] + training_confs

training_envs = [
  Simulator(*conf) for conf in training_confs
]

evaluation_envs = [
  Simulator(*conf) for conf in evaluation_confs
]

In [ ]:
for conf_idx, conf in enumerate(training_confs):
  print("Training[%s] = %s" % (conf_idx, tuple(conf)))
for conf_idx, conf in enumerate(evaluation_confs):
  print("Evaluation[%s] = %s" % (conf_idx, tuple(conf)))

### Setup of ML Models

For each one of the training environments a peer model is setup.
Then a central coordinator is setup.
Each one of the peers is loaded with the initial model contained in the coordinator, so that the federated learning league is consistent.

Then a centralized model is initialized with the same networks as the central coordinator of before so that they are comparable.

In [ ]:
centralized_model = Model.create(n_observations=training_envs[0].n_state, n_actions=training_envs[0].n_action)
federated_coordinator = Model.create(n_observations=training_envs[0].n_state, n_actions=training_envs[0].n_action)
federated_peers = [
  Model.create(n_observations=env.n_state, n_actions=env.n_action)
  for env in training_envs
]
centralized_model.load_state_of(federated_coordinator)
for peer in federated_peers:
  peer.load_state_of(federated_coordinator)

### Setup of Tracking for Models Training and Evaluation

For each model a pair of `History` instances are setup. One for training and one for evaluation.

The expected behavior is that:

- if the model learns, at the end of each episode (or episode-set, in the case of `learn_ex`) a new record is added with the total_reward accumulated in that episode (or episode-set).
- if the model is evaluated, at the end of each episode-set a new record is added with the total_reward accumulated in that episode-set.

In particular, it is crucial that the number of training records among the models is equal and that the number of evaluation records among the models is equal. Now, since the coordinator is not trained directly, it has not training history but it has an evaluation history.

In [ ]:
histories = {
    "CL": {"training": History(), "evaluation": History()},
    "FC": {"evaluation": History()},
}
for peer_idx in range(len(federated_peers)):
  histories["FP%s" % peer_idx] = {"training": History(), "evaluation": History()}

### Initial Evaluation of the models

This pass is done in order to add an initial reference record inside their evaluation history.
The expected behavior is that all the models initially start with the same state, therefore their performance should be similar.
The simulations are randomized, and the action of DQNs is non deterministic, so they won't be perfectly equal to each other if run independently. To ensure the starting point of the metric is comparable, one of the models is evaluated and its core is shared among the other models.

In [ ]:
total_reward = evaluate_ex(evaluation_envs, federated_coordinator.predict).total_reward[-1]
histories["FC"]["evaluation"].record(total_reward)
histories["CL"]["evaluation"].record(total_reward)
for peer_idx, model in enumerate(federated_peers):
  peer_ID = "FP%s" % peer_idx
  histories[peer_ID]["evaluation"].record(total_reward)

### Training and Evaluation Cycle

For each episode:
- Each peer model is trained on its own environment and then is evaluated on all the evaluation environments
- The coordinator is updated with the average of the peers' models and then is evaluated on all the evaluation environments
- Note that since every peer is trained for 1 episode, the weight of each peer is exactly the same, equal to $\frac 1 N$, where $N$ is the number of peers
- The coordinator's model is then shared among the peers
- The centralized model is trained on each one of the training environments and then is evaluated on all the evaluation environments

In [ ]:
total_vehicles_of_peers = [env.total_vehicles() for env in training_envs]
total_number_of_vehicles_among_peers = sum(total_vehicles_of_peers)
assert total_number_of_vehicles_among_peers > 0
share_of_peers = [value / total_number_of_vehicles_among_peers for value in total_vehicles_of_peers]

In [ ]:
num_of_episodes = 6
for episode_idx in range(1, num_of_episodes + 1):
  for peer_idx, (model, env) in enumerate(zip(federated_peers, training_envs)):
    peer_ID = "FP%s" % peer_idx
    total_reward = learn(env, model).total_reward[-1]
    histories[peer_ID]["training"].record(total_reward)

  for peer_idx, model in enumerate(federated_peers):
    peer_ID = "FP%s" % peer_idx
    total_reward = evaluate_ex(evaluation_envs, model.predict).total_reward[-1]
    histories[peer_ID]["evaluation"].record(total_reward)
    print("Episode", episode_idx, peer_ID, "Reward", total_reward)

  federated_coordinator.import_average_of(list(zip(share_of_peers, federated_peers)))
  total_reward = evaluate_ex(evaluation_envs, federated_coordinator.predict).total_reward[-1]
  histories["FC"]["evaluation"].record(total_reward)
  print("Episode", episode_idx, "FC", "Reward", total_reward)
  for peer in federated_peers:
    peer.load_state_of(federated_coordinator)

  total_reward = learn_ex(training_envs, centralized_model).total_reward[-1]
  histories["CL"]["training"].record(total_reward)
  total_reward = evaluate_ex(evaluation_envs, centralized_model.predict).total_reward[-1]
  histories["CL"]["evaluation"].record(total_reward)
  print("Episode", episode_idx, "CL", "Reward", total_reward)

### Saving the Models

The state and history of each model is saved to file system.

In [ ]:
models_dir = "./models"
os.makedirs(models_dir, exist_ok=True)

federated_dir = os.path.join(models_dir, "federated")
os.makedirs(federated_dir, exist_ok=True)
for idx, model in enumerate(federated_peers):
  peer_ID = "FP%s" % idx
  peer_dir = os.path.join(federated_dir, peer_ID)
  os.makedirs(peer_dir, exist_ok=True)
  model.save(peer_dir)

coordinator_dir = os.path.join(federated_dir, "FC")
os.makedirs(coordinator_dir, exist_ok=True)
federated_coordinator.save(coordinator_dir)

centralized_dir = os.path.join(models_dir, "centralized")
model_dir = os.path.join(centralized_dir, "CL")
os.makedirs(model_dir, exist_ok=True)
centralized_model.save(model_dir)

In [ ]:
histories_dir = "./histories"
os.makedirs(histories_dir, exist_ok=True)

federated_dir = os.path.join(histories_dir, "federated")
os.makedirs(federated_dir, exist_ok=True)
for idx, model in enumerate(federated_peers):
  peer_ID = "FP%s" % idx
  peer_dir = os.path.join(federated_dir, peer_ID)
  os.makedirs(peer_dir, exist_ok=True)
  histories[peer_ID]["training"].to_df().to_csv(os.path.join(peer_dir, "training.csv"))
  histories[peer_ID]["evaluation"].to_df().to_csv(os.path.join(peer_dir, "evaluation.csv"))

coordinator_dir = os.path.join(federated_dir, "FC")
os.makedirs(coordinator_dir, exist_ok=True)
histories["FC"]["evaluation"].to_df().to_csv(os.path.join(coordinator_dir, "evaluation.csv"))

centralized_dir = os.path.join(histories_dir, "centralized")
model_dir = os.path.join(centralized_dir, "CL")
os.makedirs(model_dir, exist_ok=True)
histories["CL"]["training"].to_df().to_csv(os.path.join(model_dir, "training.csv"))
histories["CL"]["evaluation"].to_df().to_csv(os.path.join(model_dir, "evaluation.csv"))

### Plotting Training Evolution

Extract the training history of the centralized model and the one of each of the peers.
The plot a graph showing for each one of the above mentioned models a curve of its training history.

In [ ]:
!mkdir -p ./figures/

In [ ]:
figure = matplotlib.pyplot.figure(figsize=(10,5))
for idx in range(len(federated_peers)):
  peer_ID = "FP%s" % idx
  df = pandas.read_csv("./histories/federated/%s/training.csv" % peer_ID)
  matplotlib.pyplot.plot(df["Episode"], df["TotalReward"], label=peer_ID, marker='o')
central_ID = "CL"
df = pandas.read_csv("./histories/centralized/%s/training.csv" % central_ID)
matplotlib.pyplot.plot(df["Episode"], df["TotalReward"], label=central_ID, marker='o')
matplotlib.pyplot.legend()
matplotlib.pyplot.title('Federated Peers vs Centralized Model')
matplotlib.pyplot.savefig('./figures/models-training-stats.png')

### Plotting Evaluation Evolution

Extract the training history of the centralized model, of the coordinator and the one of each of the peers.
The plot a graph showing for each one of the above mentioned models a curve of its training history.
Moreover, plot a graph only the coordinator and the centralized model to show their evolution.

In [ ]:
figure = matplotlib.pyplot.figure(figsize=(10,5))
for idx in range(len(federated_peers)):
  peer_ID = "FP%s" % idx
  df = pandas.read_csv("./histories/federated/%s/evaluation.csv" % peer_ID)
  matplotlib.pyplot.plot(df["Episode"], df["TotalReward"], label=peer_ID, marker='o')
coordinator_ID = "FC"
df = pandas.read_csv("./histories/federated/%s/evaluation.csv" % coordinator_ID)
matplotlib.pyplot.plot(df["Episode"], df["TotalReward"], label=coordinator_ID, marker='o')
central_ID = "CL"
df = pandas.read_csv("./histories/centralized/%s/evaluation.csv" % central_ID)
matplotlib.pyplot.plot(df["Episode"], df["TotalReward"], label=central_ID, marker='o')
matplotlib.pyplot.legend()
matplotlib.pyplot.title('Federated Peers and Coordinator vs Centralized Model')
matplotlib.pyplot.savefig('./figures/models-evaluation-stats.png')

In [ ]:
figure = matplotlib.pyplot.figure(figsize=(10,5))
coordinator_ID = "FC"
df = pandas.read_csv("./histories/federated/%s/evaluation.csv" % coordinator_ID)
matplotlib.pyplot.plot(df["Episode"], df["TotalReward"], label=coordinator_ID, marker='o')
central_ID = "CL"
df = pandas.read_csv("./histories/centralized/%s/evaluation.csv" % central_ID)
matplotlib.pyplot.plot(df["Episode"], df["TotalReward"], label=central_ID, marker='o')
matplotlib.pyplot.legend()
matplotlib.pyplot.title('Federated Coordinator vs Centralized Model')
matplotlib.pyplot.savefig('./figures/federated-coordinator-catching-up.png')

# Comparing with Classical Traffic Light Management

In [ ]:
for id, CTLA in CTLAs.items():
  print(id, CTLA)

In [ ]:
CTLA_episodes = {}
for id, CTLA in CTLAs.items():
  CTLA_episodes[id] = evaluate_ex(evaluation_envs, CTLA.choose)

In [ ]:
figure = matplotlib.pyplot.figure(figsize=(10,5))
CTLA_data_X = [id for id in CTLA_episodes]
CTLA_data_Y = [CTLA_episodes[id].total_reward[-1] for id in CTLA_data_X]
matplotlib.pyplot.ylim(-250, -100)
matplotlib.pyplot.bar(CTLA_data_X, CTLA_data_Y)
matplotlib.pyplot.legend()
matplotlib.pyplot.title('Classical Traffic Light Management Models')
matplotlib.pyplot.savefig('./figures/CTLA-10-is-best-60-is-worst.png')

In [ ]:
histories_dir = "./histories"
classical_dir = os.path.join(histories_dir, "ctla")
os.makedirs(classical_dir, exist_ok=True)
for agent_ID, agent_episode in CTLA_episodes.items():
  agent_dir = os.path.join(classical_dir, agent_ID)
  os.makedirs(agent_dir, exist_ok=True)
  agent_episode.to_df().to_csv(os.path.join(agent_dir, "evaluation.csv"))

In [ ]:
figure = matplotlib.pyplot.figure(figsize=(10,5))
coordinator_ID = "FC"
df = pandas.read_csv("./histories/federated/%s/evaluation.csv" % coordinator_ID)
matplotlib.pyplot.plot(df["Episode"], df["TotalReward"], label=coordinator_ID, marker='o')
central_ID = "CL"
df = pandas.read_csv("./histories/centralized/%s/evaluation.csv" % central_ID)
matplotlib.pyplot.plot(df["Episode"], df["TotalReward"], label=central_ID, marker='o')
number_of_episodes = len(df["Episode"])
for id, episode in CTLA_episodes.items():
  matplotlib.pyplot.plot(list(range(1, number_of_episodes+1)), [episode.total_reward[-1]] * number_of_episodes, label=id, marker='o')
matplotlib.pyplot.title('Federated Coordinator vs Centralized Model vs CTLAs')
matplotlib.pyplot.legend()
matplotlib.pyplot.savefig('./figures/FedRL-and-CRL-Learned-a-Better-Policy-then-CTLA.png')

In [ ]:
figure = matplotlib.pyplot.figure(figsize=(10,5))
coordinator_ID = "FC"
df = pandas.read_csv("./histories/federated/%s/evaluation.csv" % coordinator_ID)
matplotlib.pyplot.plot(df["Episode"], df["TotalReward"], label=coordinator_ID, marker='o')
central_ID = "CL"
df = pandas.read_csv("./histories/centralized/%s/evaluation.csv" % central_ID)
matplotlib.pyplot.plot(df["Episode"], df["TotalReward"], label=central_ID, marker='o')
number_of_episodes = len(df["Episode"])
matplotlib.pyplot.plot(list(range(1, number_of_episodes+1)), [CTLA_episodes['CTLA_10s'].total_reward[-1]] * number_of_episodes, label='CTLA_10s', marker='o')
matplotlib.pyplot.plot(list(range(1, number_of_episodes+1)), [CTLA_episodes['CTLA_60s'].total_reward[-1]] * number_of_episodes, label='CTLA_60s', marker='o')
matplotlib.pyplot.title('Federated Coordinator vs Centralized Model vs B/W CTLAs')
matplotlib.pyplot.legend()
matplotlib.pyplot.savefig('./figures/FedRL-and-CRL-Learned-a-Better-Policy-then-B_W_CTLA.png')

### Simulate Behavior and Produce Gifs

In [ ]:
federated_coordinator = Model.create(n_observations=training_envs[0].n_state, n_actions=training_envs[0].n_action)
federated_coordinator.load("./models/federated/FC")
centralized_model = Model.create(n_observations=training_envs[0].n_state, n_actions=training_envs[0].n_action)
centralized_model.load("./models/centralized/CL")

gif_dir = "./gifs"
os.makedirs(gif_dir, exist_ok=True)
federated_dir = os.path.join(gif_dir, "federated")
os.makedirs(federated_dir, exist_ok=True)
centralized_dir = os.path.join(gif_dir, "centralized")
os.makedirs(centralized_dir, exist_ok=True)

demo_env = Simulator(0.5, 0.5, 0.5, 0.5)

trace_demo(demo_env, federated_coordinator.predict)
demo_env.W.analyzer.network_fancy(file_name=os.path.join(federated_dir, "FC.gif"), sample_ratio=1, interval=5)

trace_demo(demo_env, centralized_model.predict)
demo_env.W.analyzer.network_fancy(file_name=os.path.join(centralized_dir, "CL.gif"), sample_ratio=1, interval=5)

In [ ]:
best_ctla = CTLAs['CTLA_10s']
worst_ctla = CTLAs['CTLA_60s']

gif_dir = "./gifs"
os.makedirs(gif_dir, exist_ok=True)
CTLA_dir = os.path.join(gif_dir, "ctla")
os.makedirs(CTLA_dir, exist_ok=True)

demo_env = Simulator(0.5, 0.5, 0.5, 0.5)

trace_demo(demo_env, best_ctla.choose)
demo_env.W.analyzer.network_fancy(file_name=os.path.join(CTLA_dir, "CTLA_10s.gif"), sample_ratio=1, interval=5)

trace_demo(demo_env, worst_ctla.choose)
demo_env.W.analyzer.network_fancy(file_name=os.path.join(CTLA_dir, "CTLA_60s.gif"), sample_ratio=1, interval=5)

### Show Model Behavior

In [ ]:
from IPython.display import display, Image
with open("./gifs/federated/FC.gif", "rb") as f:
    display(Image(data=f.read(), format='png'))

In [ ]:
from IPython.display import display, Image
with open("./gifs/centralized/CL.gif", "rb") as f:
    display(Image(data=f.read(), format='png'))

In [ ]:
from IPython.display import display, Image
with open("./gifs/ctla/CTLA_10s.gif", "rb") as f:
    display(Image(data=f.read(), format='png'))

In [ ]:
from IPython.display import display, Image
with open("./gifs/ctla/CTLA_60s.gif", "rb") as f:
    display(Image(data=f.read(), format='png'))

### Inspect Data

In [ ]:
for id, episode in CTLA_episodes.items():
  print(id, ':', episode.total_reward[-1])

In [ ]:
for key in histories:
  print(key)
  if "training" in histories[key]:
    print(histories[key]["training"].to_df())

In [ ]:
for key in histories:
  print(key)
  if "evaluation" in histories[key]:
    print(histories[key]["evaluation"].to_df())

### Pack and Ship Everything

In [ ]:
!rm -rf ./shipment.tar ./shipment.tar.gz
!tar cvf ./shipment.tar ./models ./histories ./gifs ./figures
!gzip ./shipment.tar

In [ ]:
!du -hs shipment.tar.gz